In [1]:
#@title Install Packages
%%capture
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install rdkit torch_geometric

In [2]:
#@title Import Packages
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

from tqdm.auto import tqdm

tqdm.pandas()
import torch
from torch_geometric.datasets import QM9
from torch_geometric.data import Data

In [3]:
#@title Define Functions
%%capture

# RDkit
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors

def visualize_rdkit_mol_2d(mol):
  from rdkit.Chem.Draw import MolToImage
  from rdkit import Chem
  from rdkit.Chem import rdDepictor

  # Attempt to sanitize and kekulize the molecule for better depiction
  # It's good practice to make a copy if you don't want to modify the original supplier object's molecule
  sanitized_mol = Chem.Mol(mol) # Create a copy
  try:
      Chem.SanitizeMol(sanitized_mol)
      # Kekulize for clearer ring depictions, if applicable
      Chem.Kekulize(sanitized_mol)
  except Exception as e:
      print(f"Warning: Could not sanitize or kekulize molecule at index {mol_index}: {e}")
      sanitized_mol = mol # Fallback to original if sanitization fails

  # Compute 2D coordinates for the (potentially) sanitized molecule
  rdDepictor.Compute2DCoords(sanitized_mol)

  # Visualize the molecule
  img = MolToImage(sanitized_mol, size=(300, 300))
  display(img)


import torch
from torch_geometric.data import Data

ATOM_FEATURES = {
    "atomic_num":       list(range(1, 10)),
    "degree":           [0, 1, 2, 3, 4],
    "chiral_tag":       [0, 1, 2],
    "hybridization":    [
        Chem.rdchem.HybridizationType.SP,
        Chem.rdchem.HybridizationType.SP2,
        Chem.rdchem.HybridizationType.SP3,
        Chem.rdchem.HybridizationType.SP3D,
        Chem.rdchem.HybridizationType.SP3D2,
    ],
}

BOND_FEATURES = {
    "bond_type": [
        Chem.rdchem.BondType.SINGLE,
        Chem.rdchem.BondType.DOUBLE,
        Chem.rdchem.BondType.TRIPLE,
        Chem.rdchem.BondType.AROMATIC,
    ],
    "stereo": [0, 1, 6, 7]
}


def one_hot(value, choices):
    """One-hot encode a value; unknown → all-zeros."""
    encoding = [0] * (len(choices) + 1)
    idx = choices.index(value) if value in choices else len(choices)
    encoding[idx] = 1
    return encoding


def atom_features_onehot(atom) -> list:
    feats = []
    feats += one_hot(atom.GetAtomicNum(),        ATOM_FEATURES["atomic_num"])
    feats += one_hot(atom.GetDegree(),           ATOM_FEATURES["degree"])
    feats += one_hot(int(atom.GetChiralTag()),   ATOM_FEATURES["chiral_tag"])
    feats += one_hot(atom.GetHybridization(),    ATOM_FEATURES["hybridization"])
    return feats


def bond_features_onehot(bond) -> list:
    feats = []
    feats += one_hot(bond.GetBondType(),         BOND_FEATURES["bond_type"])
    feats += one_hot(int(bond.GetStereo()),      BOND_FEATURES["stereo"])
    feats.append(int(bond.GetIsConjugated()))
    feats.append(int(bond.IsInRing()))
    return feats


def atom_features_int(atom) -> list:
    feats = []
    feats.append(int(atom.GetAtomicNum()))
    feats.append(int(atom.GetDegree()))
    feats.append(int(atom.GetChiralTag()))
    feats.append(int(atom.GetHybridization()))
    return feats


def bond_features_int(bond) -> list:
    feats = []
    feats.append(int(bond.GetBondType()))
    feats.append(int(bond.GetStereo()))
    feats.append(int(bond.GetIsConjugated()))
    feats.append(int(bond.IsInRing()))
    return feats



def mol_to_graph(mol, y=None, one_hot=True):

    if mol is None:
        return None

    if one_hot:
        atom_features = atom_features_onehot
        bond_features = bond_features_onehot
    else:
        atom_features = atom_features_int
        bond_features = bond_features_int

    # Atom features
    x = torch.tensor(
        [atom_features(a) for a in mol.GetAtoms()], dtype=torch.float
    )

    # Bond features (undirected => two directed edges per bond)
    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        edge_index += [[i, j], [j, i]]
        edge_attr  += [bf, bf]

    if len(edge_index) == 0:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr  = torch.zeros((0, len(bond_features(
            mol.GetBondWithIdx(0)
        ))), dtype=torch.float) if mol.GetNumBonds() > 0 else torch.zeros((0, 14), dtype=torch.float)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
        edge_attr  = torch.tensor(edge_attr,  dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
    if y is not None:
        data.y = torch.tensor([y], dtype=torch.float)
    return data

def smi_to_graph(smi, y=None):
    mol = Chem.MolFromSmiles(smi)
    return mol_to_graph(mol, y)

def visualize_graph(mol_graph):
  from torch_geometric.utils import to_networkx
  import networkx as nx
  import matplotlib.pyplot as plt
  import torch

  data_instance = mol_graph

  # Convert the PyG graph to a NetworkX graph
  G = to_networkx(data_instance, to_undirected=True)

  # Define mapping for atomic numbers to element symbols
  atomic_number_map = {
      1: 'H', 2: 'He', 3: 'Li', 4: 'Be', 5: 'B',
      6: 'C', 7: 'N', 8: 'O', 9: 'F', 14: 'Si', 15: 'P',
      16: 'S', 17: 'Cl', 35: 'Br', 53: 'I'
  }

  # Prepare node labels (atom types)
  node_labels = {}
  # The length of the one-hot vector for atomic_num is len(ATOM_FEATURES["atomic_num"]) + 1
  atomic_num_choices = ATOM_FEATURES["atomic_num"]
  one_hot_vec_len = len(atomic_num_choices) + 1

  for i, node_features_tensor in enumerate(data_instance.x):
      # Extract the one-hot encoded atomic number part
      atomic_num_one_hot_vector = node_features_tensor[:one_hot_vec_len]

      # Find the index of the '1' in the one-hot vector
      atomic_num_idx = torch.argmax(atomic_num_one_hot_vector).item()

      # Convert the index back to the actual atomic number
      if atomic_num_idx < len(atomic_num_choices):
          atomic_num = atomic_num_choices[atomic_num_idx]
      else:
          # This means the atomic number was not in the choices list (e.g., if one_hot encoded it as 'unknown')
          atomic_num = 0 # Represent as unknown or fallback

      node_labels[i] = atomic_number_map.get(atomic_num, f'Unknown({atomic_num})')

  # Define mapping for edge attributes (bond types)
  # Assuming edge_attr is a one-hot encoding for [single, double, triple, aromatic]
  bond_type_map = {
      0: 'single',
      1: 'double',
      2: 'triple',
      3: 'aromatic'
  }

  # Prepare edge labels
  edge_labels = {}
  # Iterate through the original PyG edge_index and edge_attr to get labels
  # edge_index.t() transposes (2, num_edges) to (num_edges, 2) for easier iteration
  for i, (u, v) in enumerate(data_instance.edge_index.t().tolist()):
      # Get the one-hot encoded attribute vector for the current edge
      attr_vector = data_instance.edge_attr[i]

      # Determine the bond type from the one-hot vector
      # Only process if the vector has a '1' (i.e., it's a valid one-hot)
      if attr_vector.sum() > 0:
          bond_type_idx = torch.argmax(attr_vector).item()
          label = bond_type_map.get(bond_type_idx, f"type_{bond_type_idx}")
      else:
          label = "unknown" # Fallback for unexpected attribute vectors

      # Add the label to the dictionary. NetworkX's draw_networkx_edge_labels
      # will correctly match this label to the undirected edge in G.
      # We'll add it for the (u, v) pair as it appears in edge_index.
      edge_labels[(u, v)] = label


  plt.figure(figsize=(6, 6)) # Make figure slightly larger to accommodate labels
  pos = nx.spring_layout(G, seed=42) # Use a fixed seed for reproducible layout

  # Draw nodes and edges
  nx.draw_networkx(G, pos,
                  labels=node_labels, # Use atom types as node labels
                  node_color='skyblue',
                  node_size=700, # Slightly larger nodes
                  font_size=10,
                  font_weight='bold',
                  width=1.5) # Slightly thicker edges

  # Draw edge labels
  nx.draw_networkx_edge_labels(G, pos,
                                edge_labels=edge_labels,
                                font_color='red',
                                font_size=8)

  plt.title(f"Graph Visualization")
  plt.axis('off') # Hide axes for cleaner look
  plt.show()

In [4]:
#@title Load Model and Training
class GIN(torch.nn.Module):
    """Graph Isomorphism Network class with 3 GINConv layers and 2 linear layers"""

    def __init__(self, dim_input, dim_hidden):
        """Initializing GIN class

        Args:
            dim_hidden (int): the dimension of hidden layers
        """
        super(GIN, self).__init__()
        self.conv1 = GINConv(
            Sequential(Linear(dim_input, dim_hidden), BatchNorm1d(dim_hidden), ReLU(), Linear(dim_hidden, dim_hidden), ReLU())
        )
        self.conv2 = GINConv(
            Sequential(
                Linear(dim_hidden, dim_hidden), BatchNorm1d(dim_hidden), ReLU(), Linear(dim_hidden, dim_hidden), ReLU()
            )
        )
        self.conv3 = GINConv(
            Sequential(
                Linear(dim_hidden, dim_hidden), BatchNorm1d(dim_hidden), ReLU(), Linear(dim_hidden, dim_hidden), ReLU()
            )
        )
        self.lin1 = Linear(dim_hidden, dim_hidden)
        self.lin2 = Linear(dim_hidden, 1)

    def forward(self, data):
        x = data.x
        edge_index = data.edge_index
        batch = data.batch

        # Node embeddings
        h = self.conv1(x, edge_index)
        h = h.relu()
        h = self.conv2(h, edge_index)
        h = h.relu()
        h = self.conv3(h, edge_index)

        # Graph-level readout
        h = global_add_pool(h, batch)

        h = self.lin1(h)
        h = h.relu()
        h = Fun.dropout(h, p=0.5, training=self.training)
        h = self.lin2(h)

        return h

def training(loader, model, loss, optimizer):
    """Training one epoch

    Args:
        loader (DataLoader): loader (DataLoader): training data divided into batches
        model (nn.Module): GNN model to train on
        loss (nn.functional): loss function to use during training
        optimizer (torch.optim): optimizer during training

    Returns:
        float: training loss
    """
    model.train()

    current_loss = 0
    for d in loader:
        optimizer.zero_grad()
        d.x = d.x.float()
        out = model(d)
        l = loss(out, torch.reshape(d.y, (len(d.y), 1)))
        current_loss += l / len(loader)
        l.backward()
        optimizer.step()
    return current_loss, model

@torch.no_grad()
def validation(loader, model, loss):
    """Validation

    Args:
        loader (DataLoader): validation set in batches
        model (nn.Module): current trained model
        loss (nn.functional): loss function

    Returns:
        float: validation loss
    """
    model.eval()
    val_loss = 0
    for d in loader:
        out = model(d)
        l = loss(out, torch.reshape(d.y, (len(d.y), 1)))
        val_loss += l / len(loader)
    return val_loss


@torch.no_grad()
def testing(loader, model):
    """Testing

    Args:
        loader (DataLoader): test dataset
        model (nn.Module): trained model

    Returns:
        float: test loss
    """
    loss = torch.nn.MSELoss()
    test_loss = 0
    test_target = numpy.empty((0))
    test_y_target = numpy.empty((0))
    for d in loader:
        out = model(d)
        l = loss(out, torch.reshape(d.y, (len(d.y), 1)))
        test_loss += l / len(loader)

        # save prediction vs ground truth values for plotting
        test_target = numpy.concatenate((test_target, out.detach().numpy()[:, 0]))
        test_y_target = numpy.concatenate((test_y_target, d.y.detach().numpy()))

    return test_loss, test_target, test_y_target


def train_epochs(epochs, learning_rate, model, train_loader, val_loader, file_path="best-model.pt"):
    import math
    import time
    """Training over all epochs

    Args:
        epochs (int): number of epochs to train for
        model (nn.Module): the current model
        train_loader (DataLoader): training data in batches
        val_loader (DataLoader): validation data in batches
        file_path (string): file to save the best model

    Returns:
        array: returning train and validation losses over all epochs
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss = torch.nn.MSELoss()

    train_loss = np.empty(epochs)
    val_loss = np.empty(epochs)
    best_loss = math.inf

    start_time = time.perf_counter()

    for epoch in range(epochs):
        epoch_loss, model = training(train_loader, model, loss, optimizer)
        v_loss = validation(val_loader, model, loss)
        if v_loss < best_loss:
            torch.save(model.state_dict(), f"{file_path}")
            best_loss = v_loss
        train_loss[epoch] = epoch_loss.detach().cpu().numpy()
        val_loss[epoch] = v_loss.detach().cpu().numpy()

        # print current train and val loss
        if epoch % 2 == 0:
          current_time = time.perf_counter() - start_time
          average_time = current_time / (epoch + 1)
          print(
              f"Epoch: {epoch:2d}, Train loss: {epoch_loss.item():.4f} Val loss: {v_loss.item():.4f}, Average Time per epoch: {average_time:.2f}"
          )
    return train_loss, val_loss

def plot_loss(loss_history, loss_val_history):
  plt.plot(loss_val_history, label='Total - Validation set')
  plt.plot(loss_history, label='Total - Training set')

  plt.yscale('log')
  plt.legend()
  plt.ylabel("Loss")
  plt.xlabel("Epoch")
  plt.show()


def plot_target(model, train_loader, val_loader, test_loader=None):

  model.eval()

  train_pred = np.empty((0))
  train_true = np.empty((0))
  val_pred = np.empty((0))
  val_true = np.empty((0))
  test_pred = np.empty((0))
  test_true = np.empty((0))

  with torch.no_grad():
      for d in train_loader:
          out = model(d)
          train_pred = np.concatenate((train_pred, out.detach().cpu().numpy()[:, 0]))
          train_true = np.concatenate((train_true, d.y.detach().cpu().numpy()))
      for d in val_loader:
          out = model(d)
          val_pred = np.concatenate((val_pred, out.detach().cpu().numpy()[:, 0]))
          val_true = np.concatenate((val_true, d.y.detach().cpu().numpy()))
      if test_loader:
        for d in test_loader:
            out = model(d)
            test_pred = np.concatenate((test_pred, out.detach().cpu().numpy()[:, 0]))
            test_true = np.concatenate((test_true, d.y.detach().cpu().numpy()))


  plt.plot(train_pred,train_true,'.',label='Train set')
  plt.plot(val_pred,val_true,'.',label='Validation set')
  if test_loader:
    plt.plot(test_pred,test_true,'.',label='Test set')
  # to get a diagonal line
  diagonal_line =np.linspace(np.min(train_true),np.max(train_true),1000)
  plt.plot(diagonal_line,diagonal_line, color='red', linestyle='--')
  # plt.axline([0, 0], slope=1, color='red', linestyle='--')
  plt.legend()
  plt.xlabel("Predicted Value")
  plt.ylabel("Actual Value")
  plt.show()

  plt.plot(train_pred,train_true-train_pred,'.',label='Train set')
  plt.plot(val_pred,val_true-val_pred,'.',label='Validation set')
  plt.plot(test_pred,test_true-test_pred,'.',label='Test set')
  plt.axhline(0, color='black', linewidth=1)
  plt.legend()
  plt.xlabel("Predicted Value")
  plt.ylabel("Residual (Actual-Predicted)")
  plt.show()

  sns.kdeplot(train_true-train_pred,label='Train set')
  sns.kdeplot(val_true-val_pred,label='Train set')
  sns.kdeplot(test_true-test_pred,label='Train set')
  plt.axvline(0, color='black', linewidth=1)
  plt.legend()
  plt.show()

In [7]:
#@title Setup Model: Gap
import torch.nn.functional as Fun
from torch.nn import Linear, Sequential, BatchNorm1d, ReLU
from torch_geometric.nn import GCNConv, GINConv
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool, global_add_pool
dim_input = 26
model = GIN(dim_input,dim_hidden=64)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print("Device: ", device)
model.to(device)

# print(model)
total_params = sum(p.numel() for p in model.parameters())
# print(f"\nTotal parameters: {total_params:,}\n")

model.load_state_dict(torch.load('best-model-gap.pt', map_location=torch.device('cpu')))
model.eval()

FileNotFoundError: [Errno 2] No such file or directory: 'best-model-gap.pt'

In [ ]:
Smile_String = "" #@param {type:"string"}
ShowGraph = False #@param {type:"boolean"}
mol=Chem.MolFromSmiles(Smile_String)
graph = smi_to_graph(Smile_String)
results = model(graph).item()
print(f"Predicted Dipole Moments: {results:.4f}")
visualize_rdkit_mol_2D(mol)
if ShowGraph: visualize_graph(graph)

In [ ]:
#@title Setup Model: Homo
import torch.nn.functional as Fun
from torch.nn import Linear, Sequential, BatchNorm1d, ReLU
from torch_geometric.nn import GCNConv, GINConv
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool, global_add_pool
dim_input = 26
model = GIN(dim_input,dim_hidden=64)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print("Device: ", device)
model.to(device)

# print(model)
total_params = sum(p.numel() for p in model.parameters())
# print(f"\nTotal parameters: {total_params:,}\n")

model.load_state_dict(torch.load('best-model-homo.pt', map_location=torch.device('cpu')))
model.eval()

In [ ]:
Smile_String = "" #@param {type:"string"}
ShowGraph = False #@param {type:"boolean"}
mol=Chem.MolFromSmiles(Smile_String)
graph = smi_to_graph(Smile_String)
results = model(graph).item()
print(f"Predicted Dipole Moments: {results:.4f}")
visualize_rdkit_mol_2D(mol)
if ShowGraph: visualize_graph(graph)

In [ ]:
#@title Setup Model: Lumo
import torch.nn.functional as Fun
from torch.nn import Linear, Sequential, BatchNorm1d, ReLU
from torch_geometric.nn import GCNConv, GINConv
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool, global_add_pool
dim_input = 26
model = GIN(dim_input,dim_hidden=64)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print("Device: ", device)
model.to(device)

# print(model)
total_params = sum(p.numel() for p in model.parameters())
# print(f"\nTotal parameters: {total_params:,}\n")

model.load_state_dict(torch.load('best-model-lumo.pt', map_location=torch.device('cpu')))
model.eval()

In [ ]:
Smile_String = "" #@param {type:"string"}
ShowGraph = False #@param {type:"boolean"}
mol=Chem.MolFromSmiles(Smile_String)
graph = smi_to_graph(Smile_String)
results = model(graph).item()
print(f"Predicted Dipole Moments: {results:.4f}")
visualize_rdkit_mol_2D(mol)
if ShowGraph: visualize_graph(graph)

In [ ]:
!wget